# Projeto Fictus | Analise de Vendas - ETL Pipeline 

*A Lufi Data Consulting iniciou o Projeto Fictus estruturando uma base analítica confiável. Este pipeline consolida os dados operacionais de uma empresa de vendas online e os transforma em um modelo dimensional pronto para suportar a análise de Vendas, Logística e Finanças — as três frentes que fundamentarão a recomendação de aquisição.*

---

## Objetivo
Pipeline completo de ETL dos dados brutos do dataset público Olist, gerando as tabelas analíticas
que alimentarão os três módulos analíticos do Projeto Fictus (Vendas, Logística e Finanças).

## Fluxo
```
data/originais/              →   Extração   →   Transformação   →   data/pre-tratados/
  olist_*.csv (Kaggle)               |                 |               dim_*.csv
  sep=vírgula, decimal=ponto    Leitura e         Limpeza, tipagem,    fato_vendas.csv
                                validação         modelagem dimensional
```

## Nota Metodológica — Deslocamento Temporal
Os dados originais do dataset Olist compreendem o período **2016–2018**.
Para fins de contextualização analítica e simulação de cenário contemporâneo,
todas as colunas de data foram deslocadas **+7 anos**, resultando no período **2023–2025**.
Este ajuste é declarado explicitamente como premissa do case e não altera nenhuma
relação temporal entre os eventos — intervalos, lead times e sequências permanecem intactos. Os primeiros meses de operação (set–dez/2023) são desconsiderados por representarem rampa inicial, não operação madura.

## Tabelas geradas
| Tabela | Tipo | Descrição |
|---|---|---|
| `dim_cliente` | Dimensão | Atributos do cliente com geolocalização |
| `dim_produto` | Dimensão | Atributos do produto com categoria traduzida |
| `dim_vendedor` | Dimensão | Atributos do vendedor com geolocalização |
| `dim_geolocalizacao` | Dimensão | Coordenadas, cidade e estado por CEP |
| `dim_tempo` | Dimensão | Calendário analítico em português (granularidade diária) |
| `fato_vendas` | Fato | Transações de venda com métricas e chaves |

> **Limitações da base analítica:** o dataset não inclui custos operacionais diretos (CMV, despesas fixas, CAC). A análise de margem nos blocos subsequentes utiliza o frete como proxy de pressão de custo variável. Conclusões sobre rentabilidade líquida exigem dados complementares não disponíveis neste dataset.

---


---
## PASSO 0 — Setup do Projeto

Execute esta célula **uma única vez** para criar a estrutura de pastas do projeto na sua máquina.

> **Nota:** o script detecta automaticamente a pasta onde este notebook está salvo
> e cria a estrutura de pastas a partir dela. Não é necessário alterar nenhum caminho.

In [ ]:
import os
from pathlib import Path

# ─── Detecta a pasta raiz do projeto automaticamente ──────────────────────────
# O notebook está em notebooks/ → a raiz é um nível acima
NOTEBOOK_DIR = Path().resolve()
# Detecta a raiz do projeto subindo a hierarquia de pastas
# Funciona em qualquer estrutura: raiz/, notebooks/, notebooks/vendas/
def _find_base(start: Path) -> Path:
    for p in [start, start.parent, start.parent.parent]:
        if (p / "data").exists() or (p / "notebooks").exists():
            return p
    return start
BASE_DIR = _find_base(NOTEBOOK_DIR)

# ─── Estrutura de pastas do projeto ───────────────────────────────────────────
# Garante que as pastas de dados existam (complementar ao notebook de setup)
for pasta in ["data/originais", "data/externos", "data/pre-tratados", "exports", "summary_logs"]:
    (BASE_DIR / pasta).mkdir(parents=True, exist_ok=True)

print(f"✅ Pastas verificadas em: {BASE_DIR}")
print("   (Para criar a estrutura completa do projeto, execute 00_setup_projeto_fictus.ipynb)")

---
## PASSO 1 — Download dos Dados

### 1.1 Baixar o dataset do Kaggle

Os dados utilizados neste projeto são públicos e estão disponíveis no Kaggle:

🔗 **https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce**

**Instruções:**
1. Acesse o link acima (é necessário ter uma conta gratuita no Kaggle)
2. Clique em **Download** (botão no canto superior direito da página)
3. Extraia o arquivo `.zip` baixado
4. Copie **todos os arquivos `.csv`** para a pasta **`data/originais/`** do projeto

### 1.2 Arquivos esperados em `data/originais/`

| Arquivo | Descrição |
|---|---|
| `olist_customers_dataset.csv` | Dados dos clientes |
| `olist_orders_dataset.csv` | Pedidos |
| `olist_order_items_dataset.csv` | Itens dos pedidos |
| `olist_order_payments_dataset.csv` | Pagamentos |
| `olist_order_reviews_dataset.csv` | Avaliações |
| `olist_products_dataset.csv` | Produtos |
| `olist_sellers_dataset.csv` | Vendedores |
| `olist_geolocation_dataset.csv` | Geolocalização por CEP |
| `product_category_name_translation.csv` | Tradução das categorias |

> ⚠️ **Não renomeie os arquivos.** O ETL carrega os nomes originais do Kaggle.

### 1.3 Verificar se todos os arquivos estão presentes

In [ ]:
DIR_ORIG = BASE_DIR / "data" / "originais"

ARQUIVOS_ESPERADOS = [
    "olist_customers_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_products_dataset.csv",
    "olist_sellers_dataset.csv",
    "olist_geolocation_dataset.csv",
    "product_category_name_translation.csv",
]

print("Verificando arquivos em data/originais/...\n")
tudo_ok = True
for arquivo in ARQUIVOS_ESPERADOS:
    caminho = DIR_ORIG / arquivo
    if caminho.exists():
        tamanho = caminho.stat().st_size / 1024
        print(f"  ✅ {arquivo:<50} {tamanho:>8.1f} KB")
    else:
        print(f"  ❌ {arquivo:<50} NÃO ENCONTRADO")
        tudo_ok = False

print()
if tudo_ok:
    print("✅ Todos os arquivos encontrados. Pode prosseguir com o ETL.")
else:
    print("❌ Arquivos faltando. Copie todos os CSVs do Kaggle para data/originais/ antes de continuar.")

---
## PASSO 2 — Download dos Dados Externos (IBGE)

Execute esta célula **uma única vez** para baixar automaticamente as séries macroeconômicas do IBGE/SIDRA necessárias para os notebooks de análise.

Os dados são obtidos diretamente via API pública do SIDRA — nenhum cadastro ou chave de acesso é necessário.

| Arquivo gerado | Fonte | Descrição |
|---|---|---|
| `data/externos/ipca_mensal.csv` | SIDRA Tabela 1737 | IPCA — variação mensal (%) |
| `data/externos/desocupacao_pnadc.csv` | SIDRA Tabela 6381 | Taxa de desocupação trimestral móvel (%) |

> **Dependências:** `pandas` e `requests` — ambas já inclusas no Anaconda.  
> Se os arquivos já existirem na pasta, o script os sobrescreve com a versão mais recente.

In [ ]:
import re
import time
import requests
import pandas as pd
from pathlib import Path

# ─── Caminhos ─────────────────────────────────────────────────────────────────
try:
    _base = Path(__file__).resolve().parent
except NameError:
    _base = Path().resolve()

try:
    _base = Path(__file__).resolve().parent
except NameError:
    _base = Path().resolve()

def _find_base(start: Path) -> Path:
    for p in [start, start.parent, start.parent.parent]:
        if (p / "data").exists() or (p / "notebooks").exists():
            return p
    return start
_BASE_DIR = _find_base(_base)
_DIR_EXT  = _BASE_DIR / "data" / "externos"
_DIR_EXT.mkdir(parents=True, exist_ok=True)

# ─── Mapeamento de meses ──────────────────────────────────────────────────────
_MESES = {
    "jan": "01", "fev": "02", "mar": "03", "abr": "04", "mai": "05", "jun": "06",
    "jul": "07", "ago": "08", "set": "09", "out": "10", "nov": "11", "dez": "12",
    "janeiro": "01", "fevereiro": "02", "março": "03", "abril": "04",
    "maio": "05", "junho": "06", "julho": "07", "agosto": "08",
    "setembro": "09", "outubro": "10", "novembro": "11", "dezembro": "12",
}

def _normalizar_periodo(texto):
    """Converte rótulos do SIDRA para MM/AAAA. Trimestre móvel: pega o último mês."""
    texto = str(texto).lower().strip()
    m = re.search(r"-([a-zç]{3,9})\s+(\d{4})", texto)
    if m:
        mes = _MESES.get(m.group(1)[:3])
        if mes: return f"{mes}/{m.group(2)}"
    m = re.search(r"([a-zç]{3,9})\s+(\d{4})", texto)
    if m:
        mes = _MESES.get(m.group(1)[:3])
        if mes: return f"{mes}/{m.group(2)}"
    m = re.search(r"([a-zç]{3})/(\d{2,4})", texto)
    if m:
        mes = _MESES.get(m.group(1))
        ano = m.group(2)
        if mes:
            if len(ano) == 2: ano = f"19{ano}" if int(ano) > 50 else f"20{ano}"
            return f"{mes}/{ano}"
    return None

def _baixar_sidra(url, descricao, tentativas=3):
    """GET na API SIDRA com retry automático."""
    print(f"  Baixando: {descricao}")
    for i in range(1, tentativas + 1):
        try:
            r = requests.get(url, timeout=60)
            r.raise_for_status()
            return r.json()
        except requests.RequestException as e:
            print(f"  [tentativa {i}/{tentativas}] Erro: {e}")
            if i < tentativas: time.sleep(3)
    raise ConnectionError("Falha ao conectar na API do IBGE. Verifique sua conexão.")

def _processar_e_salvar(dados, col_periodo, col_valor, saida):
    """Transforma JSON do SIDRA em CSV no formato Periodo;Taxa esperado pelo notebook 02."""
    registros = []
    for row in dados[1:]:
        periodo = _normalizar_periodo(row.get(col_periodo, ""))
        if periodo is None: continue
        valor_raw = str(row.get(col_valor, "")).strip()
        if valor_raw in ("...", "", "-"): continue
        try:
            valor = float(valor_raw.replace(",", "."))
        except ValueError:
            continue
        registros.append({"Periodo": periodo, "Taxa": f"{valor:.2f}".replace(".", ",")})

    if not registros:
        print("  ⚠️  Nenhum dado válido — verifique a URL da API.")
        return

    pd.DataFrame(registros).to_csv(saida, index=False, sep=";", encoding="latin-1")
    print(f"  ✅ Salvo: {saida.name}  ({len(registros)} registros)")

# ─── Tabelas a baixar ─────────────────────────────────────────────────────────
SIDRA_BASE = "https://apisidra.ibge.gov.br/values"
_TABELAS = {
    "ipca_mensal.csv": {
        "descricao": "IPCA — variação mensal % (Tabela 1737, var. 2264)",
        "url":       f"{SIDRA_BASE}/t/1737/n1/all/v/2264/p/all/d/v2264%202",
        "col_periodo": "D3N",
        "col_valor":   "V",
    },
    "desocupacao_pnadc.csv": {
        "descricao": "Taxa de desocupação PNADC trimestral móvel (Tabela 6381, var. 4099)",
        "url":       f"{SIDRA_BASE}/t/6381/n1/all/v/4099/p/all/d/v4099%201",
        "col_periodo": "D3N",
        "col_valor":   "V",
    },
}

# ─── Execução ─────────────────────────────────────────────────────────────────
print(f"Destino: {_DIR_EXT}\n")
for nome, cfg in _TABELAS.items():
    dados = _baixar_sidra(cfg["url"], cfg["descricao"])
    _processar_e_salvar(dados, cfg["col_periodo"], cfg["col_valor"], _DIR_EXT / nome)

print("\nDados externos prontos. Prossiga com o PASSO 3.")

---
## PASSO 3 — Configuração do Ambiente

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from dateutil.relativedelta import relativedelta

# ─── Caminhos relativos — funcionam em qualquer máquina ───────────────────────
NOTEBOOK_DIR = Path().resolve()
# Detecta a raiz do projeto subindo a hierarquia de pastas
# Funciona em qualquer estrutura: raiz/, notebooks/, notebooks/vendas/
def _find_base(start: Path) -> Path:
    for p in [start, start.parent, start.parent.parent]:
        if (p / "data").exists() or (p / "notebooks").exists():
            return p
    return start
BASE_DIR = _find_base(NOTEBOOK_DIR)
DIR_ORIG     = BASE_DIR / "data" / "originais"
DIR_PRE      = BASE_DIR / "data" / "pre-tratados"
DIR_PRE.mkdir(parents=True, exist_ok=True)

# ─── Constante de deslocamento temporal ───────────────────────────────────────
YEARS_SHIFT = 7

def deslocar_datas(df, colunas):
    """Aplica deslocamento de YEARS_SHIFT anos em colunas de data.
    Intervalos temporais entre eventos são integralmente preservados.
    """
    for col in colunas:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')
            df[col] = df[col].apply(
                lambda x: x + relativedelta(years=YEARS_SHIFT) if pd.notna(x) else x
            )
    return df

print("Ambiente configurado.")
print(f"  Raiz do projeto : {BASE_DIR}")
print(f"  Originais       : {DIR_ORIG}")
print(f"  Pré-tratados    : {DIR_PRE}")
print(f"  Deslocamento    : +{YEARS_SHIFT} anos")

---
## PASSO 4 — Dicionários de Tradução

In [ ]:
MESES_PT = {
    1: 'Janeiro',   2: 'Fevereiro', 3: 'Março',
    4: 'Abril',     5: 'Maio',      6: 'Junho',
    7: 'Julho',     8: 'Agosto',    9: 'Setembro',
    10: 'Outubro',  11: 'Novembro', 12: 'Dezembro'
}

DIAS_SEMANA_PT = {
    'Monday'    : 'Segunda-feira',
    'Tuesday'   : 'Terça-feira',
    'Wednesday' : 'Quarta-feira',
    'Thursday'  : 'Quinta-feira',
    'Friday'    : 'Sexta-feira',
    'Saturday'  : 'Sábado',
    'Sunday'    : 'Domingo',
}

MAPA_STATUS = {
    'delivered'   : 'entregue',
    'shipped'     : 'enviado',
    'canceled'    : 'cancelado',
    'invoiced'    : 'faturado',
    'processing'  : 'em processamento',
    'unavailable' : 'indisponivel',
    'approved'    : 'aprovado',
    'created'     : 'criado',
}

MAPA_PAGAMENTO = {
    'credit_card' : 'cartao_credito',
    'boleto'      : 'boleto',
    'voucher'     : 'voucher',
    'debit_card'  : 'cartao_debito',
    'not_defined' : 'nao_definido',
}

print("Dicionários carregados.")

---
## PASSO 5 — EXTRAÇÃO

Leitura dos arquivos originais do Kaggle sem modificação.  
Separador: vírgula (`,`) | Decimal: ponto (`.`) | Encoding: UTF-8

In [ ]:
raw_clientes   = pd.read_csv(DIR_ORIG / "olist_customers_dataset.csv",             encoding="utf-8")
raw_pedidos    = pd.read_csv(DIR_ORIG / "olist_orders_dataset.csv",                 encoding="utf-8")
raw_itens      = pd.read_csv(DIR_ORIG / "olist_order_items_dataset.csv",            encoding="utf-8")
raw_pagamentos = pd.read_csv(DIR_ORIG / "olist_order_payments_dataset.csv",         encoding="utf-8")
raw_reviews    = pd.read_csv(DIR_ORIG / "olist_order_reviews_dataset.csv",          encoding="utf-8")
raw_produtos   = pd.read_csv(DIR_ORIG / "olist_products_dataset.csv",               encoding="utf-8")
raw_vendedores = pd.read_csv(DIR_ORIG / "olist_sellers_dataset.csv",                encoding="utf-8")
raw_categorias = pd.read_csv(DIR_ORIG / "product_category_name_translation.csv",   encoding="utf-8")
raw_geo        = pd.read_csv(DIR_ORIG / "olist_geolocation_dataset.csv",            encoding="utf-8")

datasets = {
    "clientes"      : raw_clientes,
    "pedidos"       : raw_pedidos,
    "itens"         : raw_itens,
    "pagamentos"    : raw_pagamentos,
    "reviews"       : raw_reviews,
    "produtos"      : raw_produtos,
    "vendedores"    : raw_vendedores,
    "categorias"    : raw_categorias,
    "geolocalizacao": raw_geo,
}

print("Arquivos carregados:\n")
for nome, df in datasets.items():
    print(f"  {nome:<18} → {len(df):>8} linhas | {len(df.columns):>2} colunas")

---

## PASSO 6 — TRANSFORMAÇÃO

### 6.1 Renomeação das colunas para português

In [ ]:
raw_clientes.rename(columns={
    "customer_id"              : "id_cliente",
    "customer_unique_id"       : "id_unico_cliente",
    "customer_zip_code_prefix" : "prefixo_cep_cliente",
    "customer_city"            : "cidade_cliente",
    "customer_state"           : "estado_cliente",
}, inplace=True)

raw_pedidos.rename(columns={
    "order_id"                      : "id_pedido",
    "customer_id"                   : "id_cliente",
    "order_status"                  : "status_pedido",
    "order_purchase_timestamp"      : "data_compra",
    "order_approved_at"             : "data_aprovacao",
    "order_delivered_carrier_date"  : "data_envio_transportadora",
    "order_delivered_customer_date" : "data_entrega_cliente",
    "order_estimated_delivery_date" : "data_previsao_entrega",
}, inplace=True)

raw_itens.rename(columns={
    "order_id"            : "id_pedido",
    "order_item_id"       : "id_item_pedido",
    "product_id"          : "id_produto",
    "seller_id"           : "id_vendedor",
    "shipping_limit_date" : "data_limite_envio",
    "price"               : "preco",
    "freight_value"       : "valor_frete",
}, inplace=True)

raw_pagamentos.rename(columns={
    "order_id"             : "id_pedido",
    "payment_sequential"   : "numero_pagamento",
    "payment_type"         : "tipo_pagamento",
    "payment_installments" : "numero_parcelas",
    "payment_value"        : "valor_pagamento",
}, inplace=True)

raw_reviews.rename(columns={
    "review_id"               : "id_review",
    "order_id"                : "id_pedido",
    "review_score"            : "nota_review",
    "review_comment_title"    : "titulo_review",
    "review_comment_message"  : "mensagem_review",
    "review_creation_date"    : "data_criacao_review",
    "review_answer_timestamp" : "data_resposta_review",
}, inplace=True)

raw_produtos.rename(columns={
    "product_id"                 : "id_produto",
    "product_category_name"      : "nome_categoria_produto",
    "product_name_lenght"        : "comprimento_nome_produto",
    "product_description_lenght" : "comprimento_descricao_produto",
    "product_photos_qty"         : "quantidade_fotos_produto",
    "product_weight_g"           : "peso_produto_g",
    "product_length_cm"          : "comprimento_produto_cm",
    "product_height_cm"          : "altura_produto_cm",
    "product_width_cm"           : "largura_produto_cm",
}, inplace=True)

raw_vendedores.rename(columns={
    "seller_id"              : "id_vendedor",
    "seller_zip_code_prefix" : "prefixo_cep_vendedor",
    "seller_city"            : "cidade_vendedor",
    "seller_state"           : "estado_vendedor",
}, inplace=True)

raw_categorias.rename(columns={
    "product_category_name"         : "nome_categoria_produto",
    "product_category_name_english" : "nome_categoria_produto_en",
}, inplace=True)

raw_geo.rename(columns={
    "geolocation_zip_code_prefix" : "prefixo_cep",
    "geolocation_lat"             : "latitude",
    "geolocation_lng"             : "longitude",
    "geolocation_city"            : "cidade",
    "geolocation_state"           : "estado",
}, inplace=True)

print("Renomeação concluída.")

### 6.2 Tipagem explícita das colunas numéricas
Garante que os CSVs gerados saiam com separador decimal **ponto** e tipos corretos,eliminando a necessidade de qualquer ajuste manual no Power BI ou Excel.

In [ ]:
# ─── fato_vendas: colunas decimais ────────────────────────────────────────────
COLS_DECIMAL_ITENS = ["preco", "valor_frete"]
for col in COLS_DECIMAL_ITENS:
    raw_itens[col] = pd.to_numeric(raw_itens[col], errors="coerce").round(2)

COLS_DECIMAL_PAG = ["valor_pagamento"]
for col in COLS_DECIMAL_PAG:
    raw_pagamentos[col] = pd.to_numeric(raw_pagamentos[col], errors="coerce").round(2)

# ─── fato_vendas: colunas inteiras ────────────────────────────────────────────
raw_pagamentos["numero_parcelas"] = pd.to_numeric(raw_pagamentos["numero_parcelas"], errors="coerce").astype("Int64")
raw_reviews["nota_review"]        = pd.to_numeric(raw_reviews["nota_review"],        errors="coerce").astype("Int64")

# ─── dim_produto: colunas inteiras ────────────────────────────────────────────
COLS_INT_PRODUTO = [
    "peso_produto_g", "comprimento_produto_cm",
    "altura_produto_cm", "largura_produto_cm",
    "comprimento_nome_produto", "comprimento_descricao_produto",
    "quantidade_fotos_produto",
]
for col in COLS_INT_PRODUTO:
    if col in raw_produtos.columns:
        raw_produtos[col] = pd.to_numeric(raw_produtos[col], errors="coerce").astype("Int64")

print("Tipagem numérica aplicada.")
print(f"\nAmostra itens: preco={raw_itens['preco'].dtype} | valor_frete={raw_itens['valor_frete'].dtype}")
print(f"Amostra pag  : numero_parcelas={raw_pagamentos['numero_parcelas'].dtype}")
print(f"Amostra review: nota_review={raw_reviews['nota_review'].dtype}")
print(f"Amostra produto: peso_produto_g={raw_produtos['peso_produto_g'].dtype}")

### 6.3 Deslocamento temporal +7 anos

In [ ]:
raw_pedidos = deslocar_datas(raw_pedidos, [
    "data_compra", "data_aprovacao",
    "data_envio_transportadora",
    "data_entrega_cliente",
    "data_previsao_entrega",
])

raw_itens   = deslocar_datas(raw_itens,   ["data_limite_envio"])
raw_reviews = deslocar_datas(raw_reviews, ["data_criacao_review", "data_resposta_review"])

print("Período após deslocamento +7 anos:\n")
print(f"  data_compra          : {raw_pedidos['data_compra'].min().date()} → {raw_pedidos['data_compra'].max().date()}")
print(f"  data_entrega_cliente : {raw_pedidos['data_entrega_cliente'].dropna().min().date()} → {raw_pedidos['data_entrega_cliente'].dropna().max().date()}")

### 6.4 Padronização de texto e traduções

In [ ]:
raw_clientes["cidade_cliente"]    = raw_clientes["cidade_cliente"].str.strip().str.lower()
raw_clientes["estado_cliente"]    = raw_clientes["estado_cliente"].str.strip().str.upper()
raw_vendedores["cidade_vendedor"] = raw_vendedores["cidade_vendedor"].str.strip().str.lower()
raw_vendedores["estado_vendedor"] = raw_vendedores["estado_vendedor"].str.strip().str.upper()
raw_geo["cidade"]                 = raw_geo["cidade"].str.strip().str.lower()
raw_geo["estado"]                 = raw_geo["estado"].str.strip().str.upper()

raw_pedidos["status_pedido"]     = raw_pedidos["status_pedido"].map(MAPA_STATUS).fillna(raw_pedidos["status_pedido"])
raw_pagamentos["tipo_pagamento"] = raw_pagamentos["tipo_pagamento"].map(MAPA_PAGAMENTO).fillna(raw_pagamentos["tipo_pagamento"])

print("Padronização concluída.")
print(f"\nStatus únicos    : {sorted(raw_pedidos['status_pedido'].unique())}")
print(f"Pagamentos únicos: {sorted(raw_pagamentos['tipo_pagamento'].unique())}")

---
## PASSO 7 — MODELAGEM DIMENSIONAL
### 7.1 dim_geolocalizacao

In [ ]:
dim_geolocalizacao = (
    raw_geo
    .groupby("prefixo_cep")
    .agg(
        latitude  = ("latitude",  "median"),
        longitude = ("longitude", "median"),
        cidade    = ("cidade",    "first"),
        estado    = ("estado",    "first"),
    )
    .reset_index()
)

print(f"dim_geolocalizacao: {len(dim_geolocalizacao)} CEPs | {len(dim_geolocalizacao.columns)} colunas")

### 7.2 dim_cliente

In [ ]:
dim_cliente = (
    raw_clientes[[
        "id_cliente", "id_unico_cliente",
        "cidade_cliente", "estado_cliente", "prefixo_cep_cliente",
    ]]
    .drop_duplicates(subset="id_cliente")
    .reset_index(drop=True)
)

dim_cliente = dim_cliente.merge(
    dim_geolocalizacao[["prefixo_cep", "latitude", "longitude"]]
    .rename(columns={"prefixo_cep": "prefixo_cep_cliente"}),
    on="prefixo_cep_cliente", how="left"
)

print(f"dim_cliente: {len(dim_cliente)} linhas | {len(dim_cliente.columns)} colunas")
print(f"Com coordenadas: {dim_cliente['latitude'].notna().sum()} ({dim_cliente['latitude'].notna().mean()*100:.1f}%)")

### 7.3 dim_produto

In [ ]:
dim_produto = (
    raw_produtos
    .merge(raw_categorias, on="nome_categoria_produto", how="left")
    .drop_duplicates(subset="id_produto")
    .reset_index(drop=True)
)

# Usa categoria em inglês como fallback quando a portuguesa está nula
dim_produto["nome_categoria_produto"] = (
    dim_produto["nome_categoria_produto"]
    .fillna(dim_produto.get("nome_categoria_produto_en", pd.NA))
    .fillna("sem_categoria")
)

print(f"dim_produto: {len(dim_produto)} linhas | {len(dim_produto.columns)} colunas")
print(f"Categorias únicas: {dim_produto['nome_categoria_produto'].nunique()}")

### 7.4 dim_vendedor

In [ ]:
dim_vendedor = (
    raw_vendedores[[
        "id_vendedor", "cidade_vendedor",
        "estado_vendedor", "prefixo_cep_vendedor",
    ]]
    .drop_duplicates(subset="id_vendedor")
    .reset_index(drop=True)
)

dim_vendedor = dim_vendedor.merge(
    dim_geolocalizacao[["prefixo_cep", "latitude", "longitude"]]
    .rename(columns={"prefixo_cep": "prefixo_cep_vendedor"}),
    on="prefixo_cep_vendedor", how="left"
)

print(f"dim_vendedor: {len(dim_vendedor)} linhas | {len(dim_vendedor.columns)} colunas")

### 7.5 dim_tempo

In [ ]:
datas = raw_pedidos["data_compra"].dropna().dt.normalize().unique()

dim_tempo = pd.DataFrame({"data": sorted(datas)})
dim_tempo["data"]          = pd.to_datetime(dim_tempo["data"])
dim_tempo["ano"]           = dim_tempo["data"].dt.year
dim_tempo["mes"]           = dim_tempo["data"].dt.month
dim_tempo["nome_mes"]      = dim_tempo["mes"].map(MESES_PT)
dim_tempo["trimestre"]     = dim_tempo["data"].dt.quarter
dim_tempo["semana_ano"]    = dim_tempo["data"].dt.isocalendar().week.astype(int)
dim_tempo["dia"]           = dim_tempo["data"].dt.day
dim_tempo["dia_semana"]    = dim_tempo["data"].dt.day_name().map(DIAS_SEMANA_PT)
dim_tempo["fim_de_semana"] = dim_tempo["data"].dt.dayofweek >= 5
dim_tempo["ano_mes"]       = dim_tempo["data"].dt.to_period("M").astype(str)
dim_tempo["id_data"]       = dim_tempo["data"].dt.strftime("%Y%m%d").astype(int)

dim_tempo = dim_tempo[[
    "id_data", "data", "ano", "mes", "nome_mes",
    "trimestre", "semana_ano", "dia", "dia_semana",
    "fim_de_semana", "ano_mes"
]]

print(f"dim_tempo: {len(dim_tempo)} linhas | {len(dim_tempo.columns)} colunas")
print(f"Período: {dim_tempo['data'].min().date()} → {dim_tempo['data'].max().date()}")

### 7.6 Pagamentos e Reviews agregados por pedido

In [ ]:
pag_principal = (
    raw_pagamentos
    .sort_values("valor_pagamento", ascending=False)
    .groupby("id_pedido").first()
    .reset_index()[["id_pedido", "tipo_pagamento", "numero_parcelas"]]
)

pag_total = (
    raw_pagamentos
    .groupby("id_pedido")["valor_pagamento"]
    .sum().reset_index()
    .rename(columns={"valor_pagamento": "valor_pagamento_total"})
)
pag_total["valor_pagamento_total"] = pag_total["valor_pagamento_total"].round(2)

pagamentos_agg = pag_principal.merge(pag_total, on="id_pedido", how="left")

reviews_agg = (
    raw_reviews
    .sort_values("data_criacao_review", ascending=False)
    .groupby("id_pedido").first()
    .reset_index()[["id_pedido", "nota_review"]]
)

print(f"Pagamentos agregados: {len(pagamentos_agg)} pedidos únicos")
print(f"Reviews agregados   : {len(reviews_agg)} pedidos com review")

### 7.7 fato_vendas

In [ ]:
fato = raw_itens.merge(raw_pedidos, on="id_pedido", how="inner")
fato = fato.merge(pagamentos_agg,  on="id_pedido", how="left")
fato = fato.merge(reviews_agg,     on="id_pedido", how="left")

fato["id_data"] = (
    fato["data_compra"].dt.normalize()
    .dt.strftime("%Y%m%d")
    .astype("Int64")
)

# Métricas derivadas
fato["valor_total_item"]  = (fato["preco"] + fato["valor_frete"]).round(2)
fato["lead_time_dias"]    = (fato["data_entrega_cliente"] - fato["data_compra"]).dt.days.astype("Int64")
fato["atraso_dias"]       = (fato["data_entrega_cliente"] - fato["data_previsao_entrega"]).dt.days.astype("Int64")
fato["entregue_no_prazo"] = fato["atraso_dias"].apply(
    lambda x: True if pd.notna(x) and x <= 0 else (False if pd.notna(x) else None)
)

fato_vendas = fato[[
    "id_pedido", "id_item_pedido", "id_cliente",
    "id_produto", "id_vendedor", "id_data",
    "data_compra", "data_aprovacao",
    "data_envio_transportadora", "data_entrega_cliente",
    "data_previsao_entrega",
    "preco", "valor_frete", "valor_total_item",
    "valor_pagamento_total", "tipo_pagamento", "numero_parcelas",
    "lead_time_dias", "atraso_dias", "entregue_no_prazo",
    "nota_review", "status_pedido",
]].copy().reset_index(drop=True)

print(f"fato_vendas: {len(fato_vendas)} linhas | {len(fato_vendas.columns)} colunas")
print(f"\nTipos das colunas numéricas:")
for col in ["preco", "valor_frete", "valor_total_item", "valor_pagamento_total",
            "numero_parcelas", "lead_time_dias", "atraso_dias", "nota_review"]:
    print(f"  {col:<30} {fato_vendas[col].dtype}")

---
## PASSO 8 — VALIDAÇÃO DO MODELO

In [ ]:
print("=" * 55)
print("VALIDAÇÃO DO MODELO DIMENSIONAL")
print("=" * 55)

orfaos_cliente  = fato_vendas[~fato_vendas["id_cliente"].isin(dim_cliente["id_cliente"])]
orfaos_produto  = fato_vendas[~fato_vendas["id_produto"].isin(dim_produto["id_produto"])]
orfaos_vendedor = fato_vendas[~fato_vendas["id_vendedor"].isin(dim_vendedor["id_vendedor"])]
orfaos_data     = fato_vendas[~fato_vendas["id_data"].isin(dim_tempo["id_data"])]

print(f"\nClientes órfãos  : {len(orfaos_cliente)}")
print(f"Produtos órfãos  : {len(orfaos_produto)}")
print(f"Vendedores órfãos: {len(orfaos_vendedor)}")
print(f"Datas órfãs      : {len(orfaos_data)}")

print("\n" + "=" * 55)
print("RESUMO DO MODELO")
print("=" * 55)
print(f"fato_vendas       : {len(fato_vendas):>8} linhas")
print(f"dim_cliente       : {len(dim_cliente):>8} linhas")
print(f"dim_produto       : {len(dim_produto):>8} linhas")
print(f"dim_vendedor      : {len(dim_vendedor):>8} linhas")
print(f"dim_geolocalizacao: {len(dim_geolocalizacao):>8} linhas")
print(f"dim_tempo         : {len(dim_tempo):>8} linhas")

print("\n" + "=" * 55)
print("MÉTRICAS DE NEGÓCIO — VALIDAÇÃO RÁPIDA")
print("=" * 55)
entregues = fato_vendas[fato_vendas["status_pedido"] == "entregue"]
print(f"Pedidos únicos         : {fato_vendas['id_pedido'].nunique():>8}")
print(f"Pedidos entregues      : {entregues['id_pedido'].nunique():>8}")
print(f"Receita total (preço)  : R$ {fato_vendas['preco'].sum():>12,.2f}")
print(f"Frete total            : R$ {fato_vendas['valor_frete'].sum():>12,.2f}")
print(f"Ticket médio           : R$ {fato_vendas['preco'].mean():>12,.2f}")
print(f"Lead time médio (dias) : {entregues['lead_time_dias'].mean():>11.1f}")
print(f"% Entregue no prazo    : {entregues['entregue_no_prazo'].mean()*100:>10.1f}%")
print(f"Nota média review      : {fato_vendas['nota_review'].mean():>11.2f}")
print(f"Período dos dados      : {fato_vendas['data_compra'].min().date()} → {fato_vendas['data_compra'].max().date()}")

---

## PASSO 9 — LOAD
### 9.1 Exportação para `data/pre-tratados/` (CSV)
Os arquivos são salvos com:- **Separador:** vírgula (`,`)- **Decimal:** ponto (`.`)- **Encoding:** UTF-8Esses parâmetros garantem compatibilidade direta com Power BI, Excel e qualquer ferramenta de análise,sem necessidade de ajuste manual de localidade.

In [ ]:
arquivos = {
    "dim_cliente.csv"        : dim_cliente,
    "dim_produto.csv"        : dim_produto,
    "dim_vendedor.csv"       : dim_vendedor,
    "dim_geolocalizacao.csv" : dim_geolocalizacao,
    "dim_tempo.csv"          : dim_tempo,
    "fato_vendas.csv"        : fato_vendas,
}

for nome, df in arquivos.items():
    df.to_csv(
        DIR_PRE / nome,
        index=False,
        encoding="utf-8",
        sep=",",       # separador vírgula — padrão internacional
        decimal=".",   # decimal ponto — elimina necessidade de ajuste no Power BI
    )
    print(f"  ✅ {nome:<28} → {len(df):>8} linhas")

print(f"\nCSVs salvos em: {DIR_PRE}")

---
## Dicionário de Dados

### fato_vendas
| Coluna | Tipo | Descrição |
|---|---|---|
| id_pedido | str | Chave do pedido |
| id_item_pedido | int | Sequencial do item no pedido |
| id_cliente | str | FK → dim_cliente |
| id_produto | str | FK → dim_produto |
| id_vendedor | str | FK → dim_vendedor |
| id_data | int | FK → dim_tempo (YYYYMMDD) |
| data_compra | datetime | Timestamp da compra |
| data_aprovacao | datetime | Timestamp da aprovação do pagamento |
| data_envio_transportadora | datetime | Timestamp de envio ao transportador |
| data_entrega_cliente | datetime | Timestamp de entrega ao cliente |
| data_previsao_entrega | datetime | Previsão de entrega no momento da compra |
| preco | float | Preço do produto (decimal ponto) |
| valor_frete | float | Valor do frete — pago pelo cliente (decimal ponto) |
| valor_total_item | float | preco + valor_frete — valor do item incluindo frete |
| valor_pagamento_total | float | Valor total do pedido inteiro (todos os itens + parcelamento) |
| tipo_pagamento | str | Meio de pagamento principal |
| numero_parcelas | int | Número de parcelas (1–24) |
| lead_time_dias | int | Dias entre compra e entrega |
| atraso_dias | int | Dias de atraso (negativo = adiantado) |
| entregue_no_prazo | bool | True se entregue antes ou na data prevista |
| nota_review | int | Nota do review (1–5) |
| status_pedido | str | Status atual do pedido |

---

*Próximo notebook: `01_EDA_viabilidade_economica.ipynb` — A empresa se financia ou consome capital à medida que cresce?*

> Esta análise faz parte do **Projeto Fictus**, conduzido pela Lufi Data Consulting. Os três módulos analíticos — Vendas, Logística e Finanças — compõem a base do Relatório de Recomendação de Aquisição.
